# SOLiD-PINN v0.8.0 — Open benchmark recovery and schema audit
Run all. This revision reuses the existing Drive cache, recognizes CP1252 as a provisional fallback, quarantines the exact invalid Nature payload from the first run, rejects non-XLSX downloads, and records candidate text labels from worksheets. The old report remains intact. Parsed does not mean model-ready.


In [ ]:
from google.colab import drive
from pathlib import Path
import sys, subprocess, json
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/SOLiD-PINN/open_benchmarks')
CACHE=ROOT/'downloads'
REPORT=ROOT/'inventory_v2.json'
REPO=Path('/content/SOLiD-PINN-v080')
REV="2ea32a06b169871407d46f1316009a49c466c596"
SOURCES=["umd","puls2024","liopa"]
if not (REPO/".git").exists():
    subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(REPO)],check=True)
subprocess.run(["git","fetch","origin",REV],cwd=REPO,check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=REPO,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()==REV
subprocess.run([sys.executable,"-m","pip","install","openpyxl==3.1.5","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","tests/test_open_benchmark_audit.py"],cwd=REPO,check=True)
SETUP_OK=True

## Acquire and inventory
If Nature Energy still returns invalid content, download Source Data Fig. 1 and Source Data Supplementary Figs. 11–27 from https://www.nature.com/articles/s41560-024-01634-3#Sec20 (or locate the Source data section). Place the genuine workbooks in the Drive cache `puls2024/processing.xlsx` and `puls2024/curves.xlsx`. Do not overwrite valid cached files. Rerun acquisition. Zenodo need not be downloaded again.


In [ ]:
assert SETUP_OK
REPORT.unlink(missing_ok=True)
cmd=[sys.executable,"scripts/audit_open_benchmarks.py","--cache",str(CACHE),"--out",str(REPORT),"--download"]
for source in SOURCES: cmd+=["--source",source]
p=subprocess.run(cmd,cwd=REPO)
if p.returncode not in (0,1) or not REPORT.is_file(): raise RuntimeError("Inventory command failed; inspect logs")
result=json.loads(REPORT.read_text())
for item in result["files"]:
    print(item["source"],item["file"],item["status"],item.get("error",""))
    profile=item.get("profile",{})
    if "rows" in profile: print("  rows:",profile["rows"],"columns:",profile["columns"],"malformed:",profile["malformed_rows"])
    for sheet in profile.get("sheets",[]): print("  sheet:",sheet["name"],"declared dimensions:",sheet["declared_rows"],sheet["declared_columns"])
print("Inventory:",REPORT)
if any(item["status"]!="inventoried" for item in result["files"]):
    print("PARTIAL/BLOCKED: not all files available or parsed.")

## Export inventory for review
The new JSON includes candidate text labels, sheet dimensions, file hashes and errors. Review before sharing; labels are not verified headers or units. No numerical traces are exported. Raw files remain in Drive.


In [ ]:
from google.colab import files
files.download(str(REPORT))